# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?

Answer: **Africa** is the fastest growing continet because its population grew 290.4% by 2022.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*  I chose** line chart**, because it is being asked that how the population has been changed over the time. To measure that from 1970 to 2020 i used line chart because it clearly show the increment in the continet over the time period. So it is comparing relative growth.

In [2]:
# Q1 Relative population growth by continent (1970 = 100)
# YOUR CODE HERE
continent_year = (
    df_long.groupby(["Continent", "Year"], as_index=False)["Population"]
    .sum()
)
base_1970 = (
    continent_year.loc[continent_year["Year"] == 1970, ["Continent", "Population"]]
    .rename(columns={"Population": "Population_1970"})
)
continent_year = continent_year.merge(base_1970, on="Continent")
continent_year["Index (1970 = 100)"] = (
    continent_year["Population"] / continent_year["Population_1970"] * 100
)

growth_2022 = continent_year[continent_year["Year"] == 2022].copy()
growth_2022["Growth (%)"] = growth_2022["Index (1970 = 100)"] - 100
fastest = growth_2022.loc[growth_2022["Growth (%)"].idxmax()]

fig = px.line(
    continent_year,
    x="Year",
    y="Index (1970 = 100)",
    color="Continent",
    markers=False,
    title="Africa's population has grown nearly fourfold since 1970",
    labels={"Index (1970 = 100)": "Population index (1970 = 100)", "Year": "Year"},
    color_discrete_sequence=px.colors.qualitative.Safe,
    template="plotly_white"
)
fig.update_layout(
    legend_title_text="Continent",
    yaxis_title="Population index (1970 = 100)",
    xaxis_title=None,
    plot_bgcolor="white"
)
fig.update_xaxes(showgrid=False, dtick=10)
fig.update_yaxes(showgrid=True, gridcolor="#EEEEEE", rangemode="tozero")
fig.add_annotation(
    x=2022, y=float(fastest["Index (1970 = 100)"]),
    text=f"Africa: +{fastest['Growth (%)']:.1f}%",
    showarrow=True, arrowhead=2, ax=-65, ay=-35
)
fig.show()

print(f"Fastest-growing continent: {fastest['Continent']}")
print(f"Relative population growth, 1970–2022: {fastest['Growth (%)']:.1f}%")

Fastest-growing continent: Africa
Relative population growth, 1970–2022: 290.4%


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*  India added the most people which is about 860 million. and it also ranked fourth in the top 15 African countries.


I chose **bar chart** for this question because countries name are very long so the horizontal bar chart is the right choice to improve the readability of chart.

In [5]:
# Q2
# YOUR CODE HERE
# Create the population change column
df["Population change 1970–2022"] = (
    df["2022 Population"] - df["1970 Population"]
)

# Select the 15 countries with the highest population increase
top15 = (
    df.nlargest(15, "Population change 1970–2022")
    .sort_values("Population change 1970–2022", ascending=True)
    .copy()
)

africa_count = int((top15["Continent"] == "Africa").sum())

print("Africa countries among the top 15:", africa_count)

display(
    top15[["Country", "Continent", "Population change 1970–2022"]]
    .sort_values("Population change 1970–2022", ascending=False)
    .reset_index(drop=True)
)


top15 = (
    df.nlargest(15, "Population change 1970–2022")
    .sort_values("Population change 1970–2022", ascending=True)
    .copy()
)
africa_count = int((top15["Continent"] == "Africa").sum())

fig = px.bar(
    top15,
    x="Population change 1970–2022",
    y="Country",
    orientation="h",
    color="Continent",
    title="India and China added the most people between 1970 and 2022",
    labels={"Population change 1970–2022": "People added (1970–2022)", "Country": ""},
    color_discrete_sequence=px.colors.qualitative.Safe,
    template="plotly_white"
)
fig.update_layout(
    xaxis_title="Population added (people)",
    yaxis_title=None,
    legend_title_text="Continent",
    plot_bgcolor="white",
    bargap=0.22
)
fig.update_xaxes(rangemode="tozero", showgrid=True, gridcolor="#EEEEEE", tickformat=",.0s")
fig.update_yaxes(showgrid=False)
fig.show()

print("Africa countries among the top 15:", africa_count)
display(top15[["Country", "Continent", "Population change 1970–2022"]].sort_values(
    "Population change 1970–2022", ascending=False
).reset_index(drop=True))

Africa countries among the top 15: 4


,Country,Continent,Population change 1970–2022
0,India,Asia,859671872
1,China,Asia,603352887
2,Pakistan,Asia,176533990
3,Nigeria,Africa,162971948
4,Indonesia,Asia,160272945
5,United States,North America,137961517
6,Brazil,South America,118943623
7,Bangladesh,Asia,103644512
8,Ethiopia,Africa,95071678
9,DR Congo,Africa,78858479


Africa countries among the top 15: 4


,Country,Continent,Population change 1970–2022
0,India,Asia,859671872
1,China,Asia,603352887
2,Pakistan,Asia,176533990
3,Nigeria,Africa,162971948
4,Indonesia,Asia,160272945
5,United States,North America,137961517
6,Brazil,South America,118943623
7,Bangladesh,Asia,103644512
8,Ethiopia,Africa,95071678
9,DR Congo,Africa,78858479


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?   
**Answer:** There is no simple one-to-one relationship between land area and population. India and Bangladesh have very large populations relative to their land area, while Russia, Canada and Australia have very large land areas relative to their populations.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose **scatter chart **because the land area and 2022 population both are continuous numerical values. i chose different colors to highligh each country so that it can be easy to track.

In [6]:
# Q3
# YOUR CODE HERE

# Q3: Land area versus 2022 population
fig = px.scatter(
    df,
    x="Area",
    y="2022 Population",
    color="Continent",
    hover_name="Country",
    log_x=True,
    log_y=True,
    opacity=0.72,
    title="Country area alone does not determine population size",
    labels={
        "Area": "Land area (km², log scale)",
        "2022 Population": "2022 population (log scale)",
        "Continent": "Continent"
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template="plotly_white"
)
fig.update_traces(marker=dict(size=9, line=dict(width=0)))
fig.update_layout(
    xaxis_title="Land area (km², logarithmic scale)",
    yaxis_title="2022 population (logarithmic scale)",
    legend_title_text="Continent",
    plot_bgcolor="white"
)
fig.update_xaxes(showgrid=True, gridcolor="#EEEEEE")
fig.update_yaxes(showgrid=True, gridcolor="#EEEEEE")
fig.show()

# Illustrative high-population and large-area observations
print("Largest populations:")
display(df.nlargest(5, "2022 Population")[["Country", "Continent", "Area", "2022 Population"]])
print("Largest land areas:")
display(df.nlargest(5, "Area")[["Country", "Continent", "Area", "2022 Population"]])

Largest populations:


,Country,Continent,Area,2022 Population
41,China,Asia,9706961,1425887337
92,India,Asia,3287590,1417173173
221,United States,North America,9372610,338289857
93,Indonesia,Asia,1904569,275501339
156,Pakistan,Asia,881912,235824862


Largest land areas:


,Country,Continent,Area,2022 Population
171,Russia,Europe,17098242,144713314
35,Canada,North America,9984670,38454327
41,China,Asia,9706961,1425887337
221,United States,North America,9372610,338289857
27,Brazil,South America,8515767,215313498


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

Answer: Density varies substantially within continents. South America has the lowest median density in this chart, while North America has a relatively high median; several continents contain countries with much higher densities than their typical country.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a **violin plot** compares the full distribution of population across continents. It shows distribution shape as well as differences between groups, which is more informative here than plotting only a mean. Population density is strongly skewed, so a logarithmic y-axis makes the spread easier to inspect; the underlying values remain unchanged.

In [7]:
# Q4
# YOUR CODE HERE

# Q4: Distribution of country-level population density by continent
continent_order = (
    df.groupby("Continent")["Density"].median()
    .sort_values()
    .index.tolist()
)
fig = px.violin(
    df,
    x="Continent",
    y="Density",
    color="Continent",
    category_orders={"Continent": continent_order},
    box=True,
    points="outliers",
    log_y=True,
    title="Population density varies widely within every continent",
    labels={"Density": "Population density (people per km², log scale)", "Continent": ""},
    color_discrete_sequence=px.colors.qualitative.Safe,
    template="plotly_white"
)
fig.update_layout(
    showlegend=False,
    xaxis_title=None,
    yaxis_title="People per km² (logarithmic scale)",
    plot_bgcolor="white"
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#EEEEEE")
fig.show()

density_summary = df.groupby("Continent")["Density"].agg(
    Median="median", Q1=lambda s: s.quantile(0.25),
    Q3=lambda s: s.quantile(0.75), Minimum="min", Maximum="max"
).sort_values("Median")
display(density_summary.round(2))

,Median,Q1,Q3,Minimum,Maximum
Continent,,,,,
South America,17.79,5.61,26.35,0.31,65.02
Africa,59.71,26.60,140.34,2.17,871.93
Oceania,78.25,25.94,162.22,3.40,603.24
Europe,106.87,63.80,228.02,3.62,18234.50
Asia,122.97,71.51,336.35,2.17,23172.27
North America,221.42,63.30,335.51,0.03,1299.26


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Answer:** some island and european countries like Ukraine,Latvia's population declined or shrank between 2000 and 2022 while most of the Asian and African countries like Qatar, Kuwait, Niger grew during 2000 and 2022.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*  I chose a **choropleth map **because it shows geographically that where population growth and decline. I chose different color scale where 0 is a mid-point and negative value will be described as decline in the population and positive values will show the increment.   

In [8]:
# Q5
# YOUR CODE HERE

# Q5: Percentage population change, 2000–2022, by country
map_df = df.copy()
map_df["Population change (%) 2000–2022"] = (
    (map_df["2022 Population"] / map_df["2000 Population"]) - 1
) * 100

fig = px.choropleth(
    map_df,
    locations="CCA3",
    locationmode="ISO-3",
    color="Population change (%) 2000–2022",
    hover_name="Country",
    hover_data={
        "CCA3": False,
        "Population change (%) 2000–2022": ":.1f",
        "2000 Population": ":,",
        "2022 Population": ":,"
    },
    color_continuous_scale="RdYlBu_r",
    color_continuous_midpoint=0,
    title="Population growth and decline show clear geographic patterns, 2000–2022",
    labels={"Population change (%) 2000–2022": "Population change (%)"},
    template="plotly_white"
)
fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True, coastlinecolor="#AAAAAA"),
    coloraxis_colorbar=dict(title="Change (%)")
)
fig.show()

print("Countries with population decline:", int((map_df["Population change (%) 2000–2022"] < 0).sum()))
print("Countries with population growth:", int((map_df["Population change (%) 2000–2022"] > 0).sum()))
print("Largest percentage declines:")
display(map_df.nsmallest(10, "Population change (%) 2000–2022")[
    ["Country", "Continent", "Population change (%) 2000–2022"]
].round(1))
print("Largest percentage increases:")
display(map_df.nlargest(10, "Population change (%) 2000–2022")[
    ["Country", "Continent", "Population change (%) 2000–2022"]
].round(1))

Countries with population decline: 34
Countries with population growth: 200
Largest percentage declines:


,Country,Continent,Population change (%) 2000–2022
153,Northern Mariana Islands,Oceania,-38.3
3,American Samoa,Oceania,-24.0
117,Lithuania,Europe,-23.6
126,Marshall Islands,Oceania,-23.3
133,Moldova,Europe,-23.0
111,Latvia,Europe,-22.6
25,Bosnia and Herzegovina,Europe,-22.6
226,Vatican City,Europe,-21.7
229,Wallis and Futuna,Oceania,-21.4
218,Ukraine,Europe,-18.8


Largest percentage increases:


,Country,Continent,Population change (%) 2000–2022
167,Qatar,Asia,317.2
219,United Arab Emirates,Asia,188.2
59,Equatorial Guinea,Africa,144.5
215,Turks and Caicos Islands,North America,143.8
148,Niger,Africa,125.5
104,Jordan,Asia,123.2
108,Kuwait,Asia,120.6
5,Angola,Africa,117.1
39,Chad,Africa,114.6
230,Western Sahara,Africa,113.0


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Answer:** Asia (China & India) contains the largest part of the 2022 world's population.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a **Treemap** because i wanted to show two  hierarchical levels according to the question.Rectangle area shows the additive 2022 population, so larger rectangles represent more people.

In [9]:
# Q6
# YOUR CODE HERE

# Q6: 2022 population hierarchy — Continent > Country
fig = px.treemap(
    df,
    path=["Continent", "Country"],
    values="2022 Population",
    color="Continent",
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Asia contains the largest share of the world's population",
    hover_data={"2022 Population": ":,"},
    template="plotly_white"
)
fig.update_traces(
    textinfo="label+percent parent",
    marker=dict(line=dict(width=1, color="white")),
    tiling=dict(pad=2)
)
fig.update_layout(
    margin=dict(t=65, l=10, r=10, b=10),
    coloraxis_showscale=False
)
fig.show()

continent_totals = (
    df.groupby("Continent", as_index=False)["2022 Population"]
    .sum()
    .sort_values("2022 Population", ascending=False)
)
continent_totals["World share (%)"] = (
    continent_totals["2022 Population"] / df["2022 Population"].sum() * 100
)
display(continent_totals.round({"World share (%)": 2}))

,Continent,2022 Population,World share (%)
1,Asia,4721383274,59.21
0,Africa,1426730932,17.89
2,Europe,743147538,9.32
3,North America,600296136,7.53
5,South America,436816608,5.48
4,Oceania,45038554,0.56


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Answer:** Africa's share grew from 9.9% in 1970 to 17.9% in 2022. Europe's population fell from about 17.8% to 9.3% and Asia remained the largest.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*   I chose a **100% stacked bar** to answer this question. At two endpoints 1970 and 2022 100% stacked bar compares the world population. Each bar is 100% of the world population, and the segments show the shares of Africa, Asia, Europe and the combined rest of the world.

In [10]:
# Q7
# YOUR CODE HERE

# Q7: World population shares in 1970 and 2022
selected = ["Africa", "Asia", "Europe"]
share_data = (
    df_long.groupby(["Year", "Continent"], as_index=False)["Population"]
    .sum()
)
world_totals = share_data.groupby("Year")["Population"].transform("sum")
share_data["Share (%)"] = share_data["Population"] / world_totals * 100

share_data["Group"] = np.where(
    share_data["Continent"].isin(selected),
    share_data["Continent"],
    "Rest of the world"
)
share_grouped = (
    share_data.groupby(["Year", "Group"], as_index=False)["Population"]
    .sum()
)
share_grouped["Share (%)"] = (
    share_grouped["Population"] /
    share_grouped.groupby("Year")["Population"].transform("sum") * 100
)
share_grouped = share_grouped[share_grouped["Year"].isin([1970, 2022])]

group_order = ["Africa", "Asia", "Europe", "Rest of the world"]
share_grouped["Group"] = pd.Categorical(
    share_grouped["Group"], categories=group_order, ordered=True
)
share_grouped = share_grouped.sort_values(["Year", "Group"])

fig = px.bar(
    share_grouped,
    x="Year",
    y="Share (%)",
    color="Group",
    barmode="stack",
    category_orders={"Group": group_order},
    color_discrete_map={
        "Africa": "#2E75B6",
        "Asia": "#E07B39",
        "Europe": "#AAAAAA",
        "Rest of the world": "#F5C842"
    },
    title="Africa's share rose while Europe's share of world population declined",
    labels={"Share (%)": "Share of world population (%)", "Year": ""},
    template="plotly_white"
)
fig.update_layout(
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    xaxis=dict(tickmode="array", tickvals=[1970, 2022]),
    xaxis_title=None,
    yaxis_title="Share of world population (%)",
    legend_title_text=None,
    plot_bgcolor="white",
    bargap=0.35
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

share_pivot = share_grouped.pivot(index="Group", columns="Year", values="Share (%)")
share_pivot["Change (percentage points)"] = share_pivot[2022] - share_pivot[1970]
display(share_pivot.round(2))

Year,1970,2022,Change (percentage points)
Group,,,
Africa,9.89,17.89,8.00
Asia,58.06,59.21,1.15
Europe,17.76,9.32,-8.44
Rest of the world,14.29,13.57,-0.72
